# Basic datasets search

In [ ]:
from erddapy import ERDDAP
import numpy as np
import pandas as pd
import xarray as xr
import datetime
from pathlib import Path
import folium

In [ ]:
e = ERDDAP(server='http://136.243.54.252:8081/erddap',
          protocol='tabledap')

In [ ]:
e.dataset_id = 'allDatasets'
df_datasets = e.to_pandas()
df_datasets = df_datasets[df_datasets.datasetID != 'allDatasets']


# Keep a susbset of useful columns
df_datasets = df_datasets[[
 'datasetID',
 'institution',
 'cdm_data_type',
 'minLongitude (degrees_east)',
 'maxLongitude (degrees_east)',
 'minLatitude (degrees_north)',
 'maxLatitude (degrees_north)',
 'minAltitude (m)',
 'maxAltitude (m)',
 'minTime (UTC)',
 'maxTime (UTC)',
]]

df_datasets['minTime (UTC)'] = pd.to_datetime(df_datasets['minTime (UTC)'])
df_datasets['maxTime (UTC)'] = pd.to_datetime(df_datasets['maxTime (UTC)'])
df_datasets["duration"] = df_datasets['maxTime (UTC)'] - df_datasets['minTime (UTC)']

In [ ]:
df_datasets

### Missions that lasted longer than 30 days

In [ ]:
datasets_long = df_datasets[df_datasets["duration"] > datetime.timedelta(days=30)][['datasetID', 'duration']]
datasets_long

### Glider missions that reached a depth of greater than 170 m in the seas northeast of Gotland

In [ ]:
min_altitude = -170 # note the sign!
min_easting = 19
min_northing = 58
max_easting = 25
max_northing = 59

        
mask = ((df_datasets['maxAltitude (m)'] < min_altitude).values 
* (df_datasets['maxLongitude (degrees_east)'] < max_easting).values 
* (df_datasets['maxLatitude (degrees_north)'] < max_northing).values
* (df_datasets['minLongitude (degrees_east)'] > min_easting).values 
* (df_datasets['minLatitude (degrees_north)'] > min_northing).values
       )
df_datasets[mask].index.values
datasets_deep_gotland = df_datasets[mask]
datasets_deep_gotland

### NOC Mission in the North Sea

In [ ]:
min_easting = -3
max_easting = 9
min_northing = 50
max_northing = 58
mask = ((df_datasets['minLongitude (degrees_east)'] < max_easting).values 
* (df_datasets['minLatitude (degrees_north)'] < max_northing).values
* (df_datasets['maxLongitude (degrees_east)'] > min_easting).values 
* (df_datasets['maxLatitude (degrees_north)'] > min_northing).values
* (df_datasets['institution'] == 'NOC').values       )
df_datasets[mask].index.values
datasets_ns_noc = df_datasets[mask]
datasets_ns_noc

### Download data from one of these searches

In [ ]:
datasets_to_download = datasets_deep_gotland.datasetID.values

In [ ]:
def _clean_dims(ds):
    if "timeseries" in ds.sizes.keys() and "obs" in ds.sizes.keys():
        ds = ds.drop_dims("timeseries")
    if "trajectory" in ds.sizes.keys() and "obs" in ds.sizes.keys():
        ds = ds.drop_dims("trajectory")
    if "obs" in ds.sizes.keys():
        ds = ds.swap_dims({"obs": "time"})
    return ds

In [ ]:
datasets = []
cache_dir = Path('erddap_cache')
if not cache_dir.exists():
    cache_dir.mkdir()
for ds_id in datasets_to_download:
    fn = f'{ds_id}.nc'
    outfile = cache_dir / fn
    if outfile.exists():
        ds = xr.open_dataset(outfile)
        datasets.append(ds)
        continue
    e.dataset_id = ds_id
    ds = e.to_xarray()
    ds = _clean_dims(ds)
    datasets.append(ds)
    try:
        ds.to_netcdf(outfile)
    except:
        print(f'failed for {ds_id}')

### Plot some downloaded data on a map

In [ ]:
def mission_map(ds_list):
    """
    Makes an interactive folium map of a glider mission. Needs to run in a jupyter notebook

    Parameters
    ----------
    ds_list: xarray.DataSet or list of xarray.DataSets
        OG1 dataset of a glider mission. can pass a list of missions

    Returns
    -------
    m: folium.Map
        A folium map with a point for each glider dive and a sub-sampled line of glider track

    Notes
    -----
    Original author: Callum Rollo
    """
    if type(ds_list) is not list:
        ds_list = [ds_list]
    ds = ds_list[0]
    df = ds.to_pandas()[["longitude", "latitude", "DIVE_NUMBER"]].groupby("DIVE_NUMBER").median()
    m = folium.Map(location=[df.latitude.mean(), df.longitude.mean()], zoom_start=10, tiles="cartodb positron")
    folium.WmsTileLayer(
        url="https://ows.emodnet-bathymetry.eu/wms",
        layers='mean_atlas_land',
        attr='EMODnet bathymetry'
    ).add_to(m)
    color_cycle = ["red", "yellow", "green", "black", "pink"]
    for i, ds in enumerate(ds_list):
        color = color_cycle[i % len(color_cycle)]
        df = ds.to_pandas()[["longitude", "latitude", "DIVE_NUMBER"]].groupby("DIVE_NUMBER").median()
        df = df.dropna()
        df_points = ds.to_pandas()[["longitude", "latitude"]].dropna()
        coordinates = [[lat, lon] for lat, lon in zip(df_points['latitude'], df_points['longitude'])]
        if len(coordinates) > 5000:
            coordinates = coordinates[::int(np.ceil(len(coordinates) / 5000))]
        df['DIVE_NUMBER'] = df.index

        folium.PolyLine(
            locations=coordinates,
            color=color,
            weight=5,
            tooltip=f"{ds.attrs['id']}",
        ).add_to(m)

        for j, row in df.iterrows():
            folium.CircleMarker(
                location=[row['latitude'], row['longitude']],
                tooltip=f"Dive {int(row['DIVE_NUMBER'])}",
                color= 'black',
                fillOpacity= 1,
                fillColor= color,
                weight = 2,
            ).add_to(m)
    return m

In [ ]:
mission_map(datasets[::5])

### Combine datasets and make an SNS plot

In [ ]:
datasets[0]

In [ ]:
ds_combi = xr.concat(datasets, dim='time')

In [ ]:
ds_combi

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.set_theme(style="dark")

df_combi = ds_combi.to_pandas()
df_combi = df_combi[df_combi.TEMP<20]
df_combi = df_combi[df_combi.CNDC>6]

x = df_combi.CNDC.values[::100]
y = df_combi.TEMP.values[::100]

f, ax = plt.subplots(figsize=(10, 10))
sns.kdeplot(x=x, y=y, levels=10, color="w", linewidths=1)
sns.histplot(x=x, y=y, bins=200, pthresh=.0001, cmap="mako")
ax.set(xlabel='CNDC', ylabel='TEMP')